# 03 — Experiment with model code directly

Run the real model code **inside this notebook's Python kernel**. This notebook
does not launch SageMaker jobs, contact AWS, register a model or require Docker.
A notebook hosted in SageMaker still runs these calls in its own kernel.

Choose a model and a small local CSV, inspect the data, then run selection →
preparation → training → prediction → post-processing. Objects and dataframes
stay available for experiments and debugging. The included CSVs are synthetic.
Results scored on this training sample are smoke checks, not held-out accuracy.

Use the supported model environment: the supplied example needs pandas,
scikit-learn, matplotlib and AutoGluon. No packages are installed here.
Adapt the visible imports/calls to your model's interfaces; do not replace
its algorithms with notebook-specific implementations.

## Choose the model and local inputs

- Locate this checkout and add its model packages to Python's import path.
- Choose the model, optional local CSV/parameter file, and training mode.
- Create a fresh local experiment folder; existing runs remain available.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
sys.path[:0] = [str(ROOT / "mlops-models/src"), str(ROOT / "mlops-models")]

MODEL_ID = "usecase1"       # usecase1: regression; usecase2: classification.
CSV_INPUT = ""             # Blank uses this model's included synthetic CSV.
PARAMETERS_FILE = ""       # Blank uses models/<MODEL_ID>/parameters.json.
TRAINING_MODE = "1"        # 1: preliminary; 2: final. Your trainer owns the behavior.
MAX_ROWS = 200             # Keep the exploratory sample small.

MODEL_SOURCE = ROOT / "mlops-models/src/mlops_models/models" / MODEL_ID
if not MODEL_SOURCE.is_dir():
    raise ValueError(f"Unknown model folder: {MODEL_SOURCE}")
RUN = ROOT / ".runs/model-experiments" / MODEL_ID / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
RUN.mkdir(parents=True)
print("Kernel:", sys.executable)
print("Model source:", MODEL_SOURCE)
print("Local outputs:", RUN)

## Inspect and edit the experiment settings

- Read the parameter file and keep changes in this experiment's copy.
- Edit the visible settings or supply local model metadata for your trainer.
- Save the effective parameters beside the experiment outputs.

In [ ]:
import pandas as pd
from IPython.display import display

parameter_source = Path(PARAMETERS_FILE).expanduser() if PARAMETERS_FILE else MODEL_SOURCE / "parameters.json"
if not parameter_source.is_absolute():
    parameter_source = ROOT / parameter_source
parameters = json.loads(parameter_source.read_text())
parameters["model_id"] = MODEL_ID
parameters["feature_selection"]["top_k"] = 4  # Change this and rerun the experiment.
metadata_input = {}                          # Or load a compatible local metadata JSON.
parameter_path = RUN / "parameters.json"
parameter_path.write_text(json.dumps(parameters, indent=2) + "\n")
print("Parameter input:", parameter_source)
display(parameters)

## Inspect a small dataframe

- Read the local CSV with string identifiers and the selected row limit.
- Check the required ID and target columns, then inspect or edit the dataframe.
- Write this experiment's raw snapshot for the model functions.

In [ ]:
csv_path = Path(CSV_INPUT).expanduser() if CSV_INPUT else ROOT / "examples/data" / parameters["example_input"]
if not csv_path.is_absolute():
    csv_path = ROOT / csv_path
df = pd.read_csv(csv_path, nrows=MAX_ROWS, dtype={parameters["primary_key"]: str},
                 keep_default_na=False, na_values=["", r"\N"])
required = {parameters["primary_key"], parameters["target_key"]}
if df.empty or not required.issubset(df.columns):
    raise ValueError(f"Need nonempty data with columns {sorted(required)}: {csv_path}")
# Explore/filter df here before passing it to the model code.
raw_directory = RUN / "raw"
raw_directory.mkdir(exist_ok=True)
df.to_csv(raw_directory / "data.csv", index=False)
print("CSV:", csv_path, "| rows / columns:", df.shape)
display(df.head())

## Import the code you want to experiment with

- Import the real base selector, preparation and training modules.
- Show their source files and the framework class behind ModelTraining.
- Keep these imports editable when testing a use-case override.

For example, import your use case's selector or trainer instead of the base
class when testing an override. After editing source, restart the kernel and
run from the top for a clean experiment. In VS Code, set a breakpoint in the
imported `.py` file and use **Debug Cell** to step into the actual function.

In [ ]:
from mlops_models.model_base.training import feature_selection, preprocess, train

for module in (feature_selection, preprocess, train):
    print(module.__name__, "->", module.__file__)
print("Training class:", train.ModelTraining.__module__, train.ModelTraining.__name__)

## Fit and inspect the selector

- Construct the real selector with the experiment settings.
- Fit it to the dataframe and retain the selector object for inspection.
- Export its selected-feature CSV and PDF through the existing output function.

In [ ]:
selector = feature_selection.FeatureSelector(parameters)
selector.fit(df)
selection_directory = feature_selection.write_container_output(selector, RUN)
selection_file = selection_directory / "selected_features.csv"
display(pd.read_csv(selection_file))
print("Selection outputs:", selection_directory)

## Run the model's training preparation

- Pass the raw snapshot, selection and local parameter file explicitly.
- Call the same preparation function used by the application.
- Inspect the prepared dataframe without changing the source CSV.

In [ ]:
preprocess.process_train(raw_directory, selection_file, RUN, metadata_input, parameter_path)
prepared_directory = RUN / "prepared"
prepared = pd.read_csv(prepared_directory / "data.csv", dtype={parameters["primary_key"]: str})
print("Prepared shape:", prepared.shape)
display(prepared.head())

## Train directly and retain the model outputs

- Select the training mode in this kernel and construct the real trainer.
- Train using the local feature and data paths; keep the trainer and metadata visible.
- Use the existing packaging function to retain parameters and metadata beside the model.

This cell performs real training in the notebook environment. Create a fresh
experiment by running from the top when repeating a fit. The example trainer's
evaluation uses the prepared training sample; it is not an independent test.

In [ ]:
if TRAINING_MODE not in {"1", "2"}:
    raise ValueError("TRAINING_MODE must be '1' or '2'.")
os.environ["MLOPS_TRAINING_MODE"] = TRAINING_MODE
training_reports = RUN / "training-reports"
training_reports.mkdir(exist_ok=True)
trainer = train.ModelTraining(selection_file, prepared_directory, metadata_input, parameters)
checkpoint, model_metadata = trainer.train_model(RUN / "checkpoint", training_reports)
model_directory = train.write_container_output(
    checkpoint, model_metadata, RUN / "model", training_reports, parameters=parameters)
print("Saved model:", model_directory)
display(model_metadata)

## Set up prediction using the saved model

- Read the parameters and metadata saved with this exact local model.
- Set MLOPS_MODEL_DIR before importing inference modules that load globals.
- Run inference preparation on the local snapshot with the model's saved selection.

In [ ]:
os.environ["MLOPS_MODEL_DIR"] = str(model_directory.resolve())
os.environ["MLOPS_PREDICTION_FORMAT"] = "csv"
os.environ["MLOPS_WRITE_RUN_REPORT"] = "0"  # No managed-job report or AWS lookup in this notebook.
saved_parameters = json.loads((model_directory / "parameters.json").read_text())
model_metadata = json.loads((model_directory / "model_metadata.json").read_text())
from mlops_models.model_base.inference import preprocess as inference_preprocess
from mlops_models.model_base.inference import predict, postprocess

inference_preprocess.preprocess_data(
    raw_directory, selection_file, RUN / "inference", model_metadata,
    model_directory / "parameters.json")
inference_directory = RUN / "inference/prepared"
inference_frame = pd.read_csv(inference_directory / "data.csv",
                              dtype={saved_parameters["primary_key"]: str})
print("Model directory:", os.environ["MLOPS_MODEL_DIR"])
display(inference_frame.head())

## Inspect predictions and write the model's files

- Load the saved checkpoint through `gluonPredictor`, imported by the base as `ModelPredictor`.
- Keep a small prediction dataframe visible for experiments.
- Let the predictor write its normal CSV outputs into this experiment's score folder.

In [ ]:
predictor = predict.ModelPredictor(model_directory, model_metadata, saved_parameters)
print("Predictor class:", type(predictor).__module__, type(predictor).__name__)
predictions = predictor.predict_with_best_model(inference_frame.head(10))
display(predictions)
score_directory = RUN / "scores"
score_directory.mkdir(exist_ok=True)
predictor.predict_file(
    inference_directory / "data.csv", model_metadata["feature_mapping"],
    saved_parameters["primary_key"], output_directory=score_directory)
predict.write_container_output(score_directory)

## Run the real post-processing function

- Pass the raw snapshot, saved selection, scores, output folder and model directory.
- Let post-processing read the model parameters and write its existing outputs.
- List the resulting files and preview the example's analysis CSV when present.

In [ ]:
analysis_directory = RUN / "analysis"
postprocess.postprocess_predictions(
    raw_directory, selection_file, score_directory, analysis_directory, model_directory,
    analysis_mode="prediction_inference")
for path in sorted(analysis_directory.rglob("*")):
    if path.is_file():
        print(path.relative_to(RUN))
analysis_csv = analysis_directory / "prediction_analysis.csv"
if analysis_csv.is_file():
    display(pd.read_csv(analysis_csv, dtype={saved_parameters["primary_key"]: str}).head())

## Continue the experiment

Inspect `df`, `selector`, `prepared`, `trainer`, `model_metadata`, `predictor`
and `predictions`. Change data/settings in cells or durable logic in the source
modules. Keep the customer's file writers and algorithms; adapt the explicit
calls to its interfaces.

Every run from the top gets a new local output folder. Nothing here approves a
Registry version or changes an already-running pipeline. Use **02** to publish
reviewed code, then **04** to start or inspect managed training/inference.